In [ ]:
%pip install -r ./requirements.txt

In [ ]:
import re
from pathlib import Path
import requests
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np

In [ ]:
# Q1:

BOOKS: dict[int, str] = {
    2701: "Moby Dick",
    1342: "Pride and Prejudice",
    84: "Frankenstein",
    1661: "The Adventures of Sherlock Holmes",
    11: "Alice's Adventures in Wonderland",
    74: "The Adventures of Tom Sawyer",
    98: "A Tale of Two Cities",
    1232: "The Prince",
    5200: "Metamorphosis",
    1952: "The Yellow Wallpaper",
}

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

def download_book(book_id: int) -> str | None:
    url: str = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    response = requests.get(url = url, timeout=20)
    if response.status_code == 200:
        response.encoding = "utf-8"
        return response.text
    return None

def remove_boilerplate(book_name: str, book_text: str) -> str:
    start_pattern: str = fr"\*\*\* START OF THE PROJECT GUTENBERG EBOOK {book_name.upper()}.*?\*\*\*"
    end_pattern: str = fr"\*\*\* END OF THE PROJECT GUTENBERG EBOOK {book_name.upper()}.*?\*\*\*"
    book_text = re.sub(r"\r\n", "\n", book_text) # replaces all windows-style line-endings to unix-like
    start_match: re.Match[str] | None = re.search(start_pattern, book_text, flags=re.IGNORECASE | re.DOTALL)
    end_match: re.Match[str] | None = re.search(end_pattern, book_text, flags=re.IGNORECASE | re.DOTALL)
    start_index = start_match.end() if start_match is not None else 0
    end_index = end_match.start() if end_match is not None else len(book_text)
    book_text = book_text[start_index:end_index]
    return book_text.strip()

def write_book_to_file(book_id: int, text: str):
    path = DATA_DIR / f"{book_id}.txt"
    path.write_text(data = text, encoding="utf-8")
    print(f"Book: {BOOKS.get(book_id)} with id: {book_id}. File Size = {round(path.stat().st_size / 1024, 2)}kB. Beggining = {text[:100].replace("\n", "")}")

DOWNLOAD_BASE_URL = 'www.gutenberg.org/ebooks'
def download_books() -> dict[int, str]:
    book_dict: dict[int,str] = {}
    for book_id, book_name in BOOKS.items():
        result = download_book(book_id=book_id)
        if result is None:
            raise ValueError(f"failure during downloading of book {book_id}")
        text = remove_boilerplate(book_name=book_name, book_text=result)
        book_dict[book_id] = text
    return book_dict
        
        
book_dict: dict[str, int] = download_books()
for book_id, book_text in book_dict.items():
    write_book_to_file(book_id, book_text)

In [ ]:
# Q2
# \b means it touches a word boundary
CONTRACTIONS = {
    r"\bdon't\b": "do not",
    r"\bcan't\b": "cannot",
    r"\bwon't\b": "will not",
    r"\bit's\b": "it is",
    r"\bi'm\b": "i am",
    r"\bi've\b": "i have",
    r"\bi'll\b": "i will",
    r"\bthat's\b": "that is",
    r"\bthere's\b": "there is",
    r"\bi'd\b": "i would"
}


def to_lower(book_text: str) -> str:
    return book_text.lower()

def normalize_contractions(book_text: str) -> str:
    for pattern, replacement in CONTRACTIONS.items():
        book_text = re.sub(pattern, replacement, book_text, flags=re.IGNORECASE)
    return book_text

def normalize_whitespaces(book_text: str) -> str:
    book_text = re.sub(r"\s+", " ", book_text)
    return book_text

def remove_special_characters(book_text: str) -> str:
    return re.sub(r"[^a-zA-Z\s]", "", book_text)

def find_ing_words(text: str) -> list[str]:
    return re.findall(r"\b[a-zA-Z]+ing\b", text)

def normalize_text(book_text: str) -> str:
    book_text = to_lower(book_text)
    book_text = normalize_whitespaces(book_text)
    book_text = normalize_contractions(book_text)
    book_text = remove_special_characters(book_text)
    return book_text

def extract_sentences_with_keyword(book_text: str, keyword: str) -> list[str]:
    pattern = rf"[^.!?]*\b{re.escape(keyword)}\b[^.!?]*[.!?][\"”’']?"
    sentences = re.findall(pattern, book_text, flags=re.IGNORECASE)
    return [normalize_whitespaces(sentence).strip() for sentence in sentences]

def find_words(book_text: str) -> list[str]:
    return re.findall(r"\b[a-zA-Z]+\b", book_text)

keyword = "monster"
for book_id, book_text in book_dict.items():
    sentences = extract_sentences_with_keyword(book_text, keyword)
    print(f"\nBook: {BOOKS.get(book_id)}")
    print(f"Sentences with '{keyword}': {len(sentences)}")
    for sentence in sentences[:3]:
        print("-", sentence)

print("\n")

for book_id, book_text in book_dict.items():
    book_text = normalize_text(book_text)
    book_dict[book_id] = book_text

for book_id, book_text in book_dict.items():
    words: list[str] = find_ing_words(text = book_text)
    print(f"Words ending with -ing in a book {BOOKS.get(book_id)}: {words[:10]}. Total {len(words)}.")

print("\n")

ing_frequency_records = []
for book_id, book_text in book_dict.items():
    ing_words = find_ing_words(book_text)
    ing_counts = Counter(ing_words)
    for word, frequency in ing_counts.most_common(20):
        ing_frequency_records.append({
            "book_id": book_id,
            "title": BOOKS.get(book_id),
            "word": word,
            "frequency": frequency
        })

word_frequency_records = []
for book_id, book_text in book_dict.items():
    words = find_words(book_text)
    word_counts = Counter(words)
    for word, frequency in word_counts.most_common(20):
        word_frequency_records.append({
            "book_id": book_id,
            "title": BOOKS.get(book_id),
            "word": word,
            "frequency": frequency
        })

word_frequency_df = pd.DataFrame(word_frequency_records)
word_frequency_df = word_frequency_df.sort_values(by="frequency", ascending=False).reset_index(drop=True)
word_frequency_df.head(20)

def visualise_top_terms(book_id: int, num_terms: int):
    if num_terms < 1:
        raise ValueError("num_terms must be >= 1")
    selected_words_df = word_frequency_df[word_frequency_df["book_id"] == book_id].sort_values(by="frequency", ascending=False).head(num_terms)
    plt.figure(figsize=(10, 5))
    plt.bar(selected_words_df["word"], selected_words_df["frequency"])
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Top {num_terms} terms in {BOOKS.get(book_id)}")
    plt.xlabel("Term")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

first_book_id = next(iter(BOOKS))
visualise_top_terms(first_book_id, num_terms=20)

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
# Q3:
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 5000000
nltk.download('punkt_tab')
nltk.download("stopwords")

def nltk_word_tokens(book_text: str) -> list[str]:
    return word_tokenize(book_text)


def nltk_sentence_tokens(book_text: str) -> list[str]:
    return sent_tokenize(book_text)

english_stopwords = set(stopwords.words("english"))
def filter_nltk_tokens(tokens: list[str]) -> list[str]:
    return [
        token.lower()
        for token in tokens
        if token.isalpha() and token.lower() not in english_stopwords
    ]

def spacy_doc(book_text: str):
    return nlp(book_text)

def spacy_word_tokens(doc) -> list[str]:
    return [token.text for token in doc]

def spacy_sentence_tokens(doc) -> list[str]:
    return [sentence.text for sentence in doc.sents]

def filter_spacy_tokens(doc) -> list[str]:
    return [token.text.lower()
        for token in doc
        if token.is_alpha and not token.is_stop]

tokenization_records = []
nltk_frequency_records = []
spacy_frequency_records = []
for book_id, book_text in book_dict.items():
    nltk_words = nltk_word_tokens(book_text)
    nltk_sentences = nltk_sentence_tokens(book_text)
    nltk_filtered = filter_nltk_tokens(nltk_words)
    nltk_counts = Counter(nltk_filtered)

    for word, frequency in nltk_counts.most_common(20):
        nltk_frequency_records.append({
            "book_id": book_id,
            "title": BOOKS.get(book_id),
            "word": word,
            "frequency": frequency,
            "library": "NLTK"
        })

    doc = spacy_doc(book_text)
    spacy_words = spacy_word_tokens(doc)
    spacy_sentences = spacy_sentence_tokens(doc)
    spacy_filtered = filter_spacy_tokens(doc)
    spacy_counts = Counter(spacy_filtered)

    for word, frequency in spacy_counts.most_common(20):
        spacy_frequency_records.append({
            "book_id": book_id,
            "title": BOOKS.get(book_id),
            "word": word,
            "frequency": frequency,
            "library": "spaCy"
        })

    tokenization_records.append({
        "book_id": book_id,
        "title": BOOKS.get(book_id),
        "nltk_word_tokens": len(nltk_words),
        "spacy_word_tokens": len(spacy_words),
        "nltk_sentence_tokens": len(nltk_sentences),
        "spacy_sentence_tokens": len(spacy_sentences),
        "nltk_tokens_after_stopword_filtering": len(nltk_filtered),
        "spacy_tokens_after_stopword_filtering": len(spacy_filtered),
    })
    
tokenization_df = pd.DataFrame(tokenization_records)
nltk_frequency_df = pd.DataFrame(nltk_frequency_records)
spacy_frequency_df = pd.DataFrame(spacy_frequency_records)
display(tokenization_df.head(20))
display(nltk_frequency_df.head(20))
display(spacy_frequency_df.head(20))

def visualise_top_nltk_tokens(book_id: int, num_terms: int = 20):
    if num_terms < 1:
        raise ValueError("num_terms must be >= 1")
    selected_df = nltk_frequency_df[nltk_frequency_df["book_id"] == book_id].sort_values(by="frequency", ascending=False).head(num_terms)
    plt.figure(figsize=(10, 5))
    plt.bar(selected_df["word"], selected_df["frequency"])
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Top {num_terms} NLTK tokens in {BOOKS.get(book_id)}")
    plt.xlabel("Token")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

visualise_top_nltk_tokens(first_book_id, num_terms=20)

In [ ]:
book_ids = list(book_dict.keys())
book_names = [BOOKS[book_id] for book_id in book_ids]
documents = [book_dict[book_id] for book_id in book_ids]

bow_vectorizer = CountVectorizer(max_features=10000,
                                stop_words="english",
                                tokenizer=word_tokenize,
                                token_pattern=None)
bow_matrix = bow_vectorizer.fit_transform(documents)
print(bow_matrix.shape)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    index=book_names,
    columns=bow_vectorizer.get_feature_names_out()
)

def show_top_terms(book_name: str, df: pd.DataFrame,  num_terms: int = 20):
    row = df.loc[book_name]
    top_terms = row.sort_values(ascending=False).head(num_terms)
    return pd.DataFrame({
        "word": top_terms.index,
        "value": top_terms.values
    })

word_count_df = show_top_terms(BOOKS.get(first_book_id), bow_df)
print("Count Vectorizer")
display(word_count_df.head(50))

tf_idf_vectorizer = TfidfVectorizer(max_features=10000,
                                    stop_words="english",
                                    tokenizer=word_tokenize,
                                    token_pattern=None)
tf_idf_matrix = tf_idf_vectorizer.fit_transform(documents)
tf_idf_df = pd.DataFrame(
    tf_idf_matrix.toarray(),
    index=book_names,
    columns=tf_idf_vectorizer.get_feature_names_out()
)

word_value_df = show_top_terms(BOOKS.get(first_book_id), tf_idf_df)
print("TF-IDF Vectorizer")
display(word_value_df.head(50))

def word2vec_sentences(documents: list[str], chunk_size: int = 50) -> list[list[str]]:
    sentences: list[list[str]] = []
    for book_text in documents:
        words: list[str] = book_text.split()
        for i in range(0, len(words), chunk_size):
            chunk: list[str] = words[i:i + chunk_size]
            sentences.append(chunk)

    return sentences

sentences = word2vec_sentences(documents=documents)
word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=1000,
    window=8,
    min_count=5,
    sg=1
)

print(f"BOW: {bow_matrix.shape[0]} documents x {bow_matrix.shape[1]} features")
print(f"TF-IDF: {tf_idf_matrix.shape[0]} documents x {tf_idf_matrix.shape[1]} features")
print(f"Word2vec: {len(word2vec_model.wv)} words x {word2vec_model.vector_size} dimensions")

def show_similar_words(word: str, model: Word2Vec, topn: int = 10) -> pd.DataFrame:
    if word not in model.wv:
        raise ValueError(f"'{word}' is not in the Word2Vec vocabulary.")

    similar_words = model.wv.most_similar(word, topn=topn)
    return pd.DataFrame(
        similar_words,
        columns=["word", "similarity"]
    )
display(show_similar_words("love", word2vec_model, topn=10))


(10, 10000)
Count Vectorizer


,word,value
0,whale,961
1,like,575
2,old,440
3,man,435
4,ye,421
5,ahab,421
6,whales,397
7,ship,384
8,sea,379
9,time,323


TF-IDF Vectorizer


,word,value
0,whale,0.478311
1,ahab,0.281744
2,whales,0.265682
3,ye,0.167305
4,sperm,0.160614
5,stubb,0.153922
6,like,0.142270
7,queequeg,0.141876
8,ship,0.137955
9,sea,0.123637


BOW: 10 documents x 10000 features
TF-IDF: 10 documents x 10000 features
Word2vec: 9719 words x 1000 dimensions


,word,similarity
0,sympathy,0.837562
1,sincerely,0.824485
2,affection,0.819887
3,generous,0.816235
4,gratitude,0.812109
5,compassion,0.811327
6,darling,0.810387
7,loving,0.808522
8,loved,0.807692
9,health,0.803885


### Q4 Summary

Bag-of-Words and TF-IDF represent each book as a document-level vector with a fixed vocabulary size. Bag-of-Words stores raw word counts, while TF-IDF stores weighted importance scores, reducing the impact of words that appear frequently across many books.

Word2Vec is different: it learns dense vectors for individual words rather than whole books. Its vectors are trained from local word contexts, so it can capture semantic similarity between words. Bag-of-Words and TF-IDF are simple and useful for classical document classification or similarity tasks, while Word2Vec is more suitable when semantic relationships between words are important.

In [71]:
# Q5:

TARGET_LABELS = {"PERSON", "GPE", "ORG", "DATE"}

def extract_entities(book_text: str, book_id: int) -> list[dict[str, str | int]]:
    records: list[dict[str, str | int]] = []
    doc = spacy_doc(book_text)
    for entity in doc.ents:
        if entity.label_ in TARGET_LABELS:
            records.append({
                "book_id": book_id,
                "title": BOOKS.get(book_id, "Unknown"),
                "entity": entity.text,
                "label": entity.label_,
            })
    return records

entity_records: list[dict[str, str | int]] = []
for book_id, book_text in book_dict.items():
    entity_records.extend(extract_entities(book_text, book_id))

entities_df = pd.DataFrame(entity_records)
def top_k_entities(entity_df : pd.DataFrame, k:int) -> pd.DataFrame:
    top_entities_df = (entity_df
            .groupby(by = ["book_id", "title", "label", "entity"])
            .size()
            .reset_index(name="frequency")
            .sort_values(by=["book_id", "label", "frequency"], ascending=[True, True, False]))
    return top_entities_df.groupby(["book_id", "label"]).head(k).reset_index(drop=True)

def top_k_entities_for_book_and_label(top_k_entity_df: pd.DataFrame, book_id: int, label:str):
    return top_k_entity_df[(top_k_entity_df["book_id"] == book_id) &(top_k_entity_df["label"] == label)].reset_index(drop = True)

top_k_entity_df = top_k_entities(entities_df, 5)
display(top_k_entities_for_book_and_label(top_k_entity_df, book_id = first_book_id, label = "PERSON"))
    

,book_id,title,label,entity,frequency
0,2701,Moby Dick,PERSON,Ahab,462
1,2701,Moby Dick,PERSON,Starbuck,191
2,2701,Moby Dick,PERSON,Moby Dick,74
3,2701,Moby Dick,PERSON,Peleg,73
4,2701,Moby Dick,PERSON,Tashtego,51


In [74]:
def extract_entity_rich_sentences(book_dict: dict[int, str], top_k: int = 5) -> pd.DataFrame:
    records: list[dict[str, str | int]] = []
    for book_id, book_text in book_dict.items():
        doc = spacy_doc(book_text)
        for sentence in doc.sents:
            sentence_entities = [
                entity
                for entity in doc.ents
                if entity.start >= sentence.start
                and entity.end <= sentence.end
                and entity.label_ in TARGET_LABELS
            ]

            if len(sentence_entities) > 0:
                records.append({
                    "book_id": book_id,
                    "title": BOOKS.get(book_id, "Unknown"),
                    "sentence": sentence.text.strip(),
                    "entity_count": len(sentence_entities),
                    "entities": "; ".join(
                        f"({entity.label_}): {entity.text}"
                        for entity in sentence_entities
                    )
                })

    sentences_df = pd.DataFrame(records)
    sentences_df = (
        sentences_df
        .sort_values(by=["book_id", "entity_count"], ascending=[True, False])
        .groupby("book_id")
        .head(top_k)
        .reset_index(drop=True)
    )
    return sentences_df

entity_rich_sentences_df = extract_entity_rich_sentences(book_dict, top_k=5)
display(entity_rich_sentences_df)

,book_id,title,sentence,entity_count,entities
0,11,Alice's Adventures in Wonderland,"‘Edwin and Morcar, the earls of Mercia and Nor...",7,(PERSON): Edwin; (ORG): Morcar; (PERSON): Merc...
1,11,Alice's Adventures in Wonderland,"Alice’s Right Foot, Esq., Hearthrug, near the ...",5,(PERSON): Alice’s Right Foot; (GPE): Esq; (PER...
2,11,Alice's Adventures in Wonderland,"London is the capital of Paris, and Paris is t...",5,(GPE): London; (GPE): Paris; (GPE): Paris; (GP...
3,11,Alice's Adventures in Wonderland,"Edwin and Morcar, the earls of Mercia and Nort...",5,(PERSON): Edwin; (ORG): Morcar; (PERSON): Merc...
4,11,Alice's Adventures in Wonderland,The long grass rustled at her feet as the Whit...,5,(PERSON): Mouse; (PERSON): Queen; (ORG): Gryph...
5,74,The Adventures of Tom Sawyer,The Model Boy The Church Choir A Side Show Res...,11,(PERSON): Huckleberry Finn; (PERSON): Tom’s Tr...
6,74,The Adventures of Tom Sawyer,"“Why, he told Jeff Thatcher, and Jeff told Joh...",8,(PERSON): Jeff Thatcher; (PERSON): Jeff; (PERS...
7,74,The Adventures of Tom Sawyer,The crowd filed up the aisles: the aged and ne...,7,(DATE): better days; (PERSON): Douglas; (DATE)...
8,74,The Adventures of Tom Sawyer,“Run for Your Life” McDougal’s Cave Inside the...,6,(PERSON): Tom; (GPE): Becky; (PERSON): the Tow...
9,74,The Adventures of Tom Sawyer,"A good, generous prayer it was, and went into ...",6,(ORG): State; (ORG): State; (GPE): the United ...


In [76]:
def compute_entity_distribution(entities_df: pd.DataFrame) -> pd.DataFrame:
    return (entities_df.groupby(["label", "title"]).size().reset_index(name = "frequency"))

entity_distribution_df = compute_entity_distribution(entities_df)
display(entity_distribution_df.pivot(values = "frequency", index = "title", columns=["label"]))

label,DATE,GPE,ORG,PERSON
title,,,,
A Tale of Two Cities,412,415,734,1908
Alice's Adventures in Wonderland,46,33,249,583
Frankenstein,272,213,121,492
Metamorphosis,59,160,31,152
Moby Dick,513,1064,1008,2272
Pride and Prejudice,450,388,480,3310
The Adventures of Sherlock Holmes,362,219,261,1371
The Adventures of Tom Sawyer,237,119,128,1689
The Prince,200,546,415,741


In [ ]:
# Q6:
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.RNN(
            input_size = 1,
            
        )